In [1]:
"""
Implement FEMA model to train on the stocks to get a finger print of each 
then use some kind of temporal model (SARIMA, ARIMA, exponential smoothing, triple exponential smoothing (seasonal & yearly)) to predict the FEMA value for next month
Train a model (example can be LSTM) to predict the "unpredicatble" movement of the stock (what isnt obtained in the FEMA data/signal)

"""

'\nImplement FEMA model to train on the stocks to get a finger print of each \nthen use some kind of temporal model (SARIMA, ARIMA, exponential smoothing, triple exponential smoothing (seasonal & yearly)) to predict the FEMA value for next month\nTrain a model (example can be LSTM) to predict the "unpredicatble" movement of the stock (what isnt obtained in the FEMA data/signal)\n\n'

In [21]:
import pandas as pd
import numpy as np
import matplotlib as plt
import pandas_datareader.data as web
from datetime import date
import statsmodels.api as sm

In [34]:
# Import relevant data
trading_1min = pd.read_csv("data/yfinance/1m_interval_trading_data.csv")
trading_2min = pd.read_csv("data/yfinance/2m_interval_trading_data.csv")
trading_1h = pd.read_csv("data/yfinance/1h_interval_trading_data.csv")

background_data = pd.read_csv("data/yfinance/ticker_background_data_df.csv")

# Daily Fama-French 5 Factors
start = "2020-01-01"
end = date.today()

ff5 = web.DataReader("F-F_Research_Data_5_Factors_2x3_daily", "famafrench", start, end)

# ff5 is a dict-like object; the actual data is in ff5[0]
factors = ff5[0]
print(factors.tail())

/var/folders/vl/1jjwbkdj1r126_dp4xttqycm0000gn/T/ipykernel_4614/49755974.py:12: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype and then call 'to_datetime'.
  ff5 = web.DataReader("F-F_Research_Data_5_Factors_2x3_daily", "famafrench", start, end)


            Mkt-RF   SMB   HML   RMW   CMA    RF
Date                                            
2026-06-24   -0.07  0.78 -0.21  0.42  0.61  0.01
2026-06-25   -0.13  0.61  0.91 -0.76  0.51  0.01
2026-06-26    0.15  1.36 -0.95  0.44  0.04  0.01
2026-06-29    1.20 -0.89 -0.90 -1.77 -0.50  0.01
2026-06-30    0.73 -0.10 -0.62 -1.10 -0.49  0.01


In [35]:
dfs = {
    "1min": trading_1min,
    "2min": trading_2min,
    "1h": trading_1h,
}

for name, df in dfs.items():
    #Convert to EST
    df['Datetime'] = pd.to_datetime(df['Datetime'], utc=True).dt.tz_convert('America/New_York')
    df['Date'] = df['Datetime'].dt.normalize()

    df.sort_values(['Ticker', 'Datetime'], inplace=True)

    daily_close = df.groupby(['Ticker', 'Date'])['Close'].last().reset_index()
    daily_close = daily_close.sort_values(['Ticker', 'Date'])

    daily_close['Daily_Return'] = daily_close.groupby('Ticker')['Close'].pct_change()
    merged = df.merge(daily_close[['Ticker', 'Date', 'Daily_Return']], on=['Ticker', 'Date'], how='inner')

    #CALCULATE RISK FREE RETURNS TO OBTAIN STOCK EXCESS RETURNS 

    # Both datasets are in EST, but strip it from base 
    merged['Date'] = merged['Date'].dt.tz_localize(None)
    merged = merged.merge(factors, left_on = 'Date', right_index=True, how='left')

    merged['Excess_Return'] = merged['Daily_Return'] - merged['RF'] #Calculate excess returns using Risk Free Returns 
    
    dfs[name] = merged  # store the result back in the dict

dfs['1h'][dfs['1h']['Ticker'] == 'AAPL'][['Date', 'Ticker', 'Daily_Return', 'Excess_Return']].drop_duplicates().head(10)

/var/folders/vl/1jjwbkdj1r126_dp4xttqycm0000gn/T/ipykernel_4614/1764502183.py:17: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  daily_close['Daily_Return'] = daily_close.groupby('Ticker')['Close'].pct_change()


,Date,Ticker,Daily_Return,Excess_Return
0,2024-08-02,AAPL,NaN,NaN
7,2024-08-05,AAPL,-0.048271,-0.068271
14,2024-08-06,AAPL,-0.009871,-0.029871
21,2024-08-07,AAPL,0.013205,-0.006795
28,2024-08-08,AAPL,0.016582,-0.003418
35,2024-08-09,AAPL,0.013875,-0.006125
42,2024-08-12,AAPL,0.005779,-0.014221
49,2024-08-13,AAPL,0.017100,-0.002900
56,2024-08-14,AAPL,0.001582,-0.018418
63,2024-08-15,AAPL,0.013943,-0.006057


In [70]:
### LOOK INTO VECTORIZING THIS AND MAKING IT SO I CAN TRAIN MULTIPLE FAMA MODELS WITH MULTIPLE TIME SLICES 

fama_models = {}

input_df = dfs['1h']

for ticker in dfs['1h']['Ticker'].unique():
    subset_x = ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']
    fama_input = input_df[input_df['Ticker'] == ticker]

    fama_input = fama_input.dropna(subset=subset_x, how='all') #Filter out for data we have FAMA on
    fama_input = fama_input[fama_input['Date'] != fama_input['Date'].min()] #Filter out the first date (no returns)

    
    if not fama_input.empty:
        X = sm.add_constant(fama_input[subset_x])
        y = fama_input['Excess_Return']
        
        fama_5_factor_model = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 5}) #LOOK INTO OBTAINING AN OPTIMAL VALUE 

        fama_models[ticker] = fama_5_factor_model
    else: 
        print(f"[SKIP] {ticker}: no valid rows after filtering")

for ticker, model in fama_models.items():
    print(f"{ticker} Model Parameters: {fama_models[ticker].params}.")

AAPL Model Parameters: const    -0.017956
Mkt-RF    0.012724
SMB      -0.001252
HML      -0.000422
RMW       0.006201
CMA       0.002300
dtype: float64.
AMZN Model Parameters: const    -0.017851
Mkt-RF    0.013217
SMB       0.001216
HML      -0.003897
RMW       0.003723
CMA      -0.005132
dtype: float64.
AXP Model Parameters: const    -0.018282
Mkt-RF    0.013764
SMB       0.003119
HML       0.006323
RMW       0.001169
CMA       0.001432
dtype: float64.
BRK-B Model Parameters: const    -0.018401
Mkt-RF    0.006573
SMB      -0.001672
HML       0.007417
RMW       0.002022
CMA      -0.000224
dtype: float64.
CAT Model Parameters: const    -0.016616
Mkt-RF    0.012600
SMB       0.004035
HML       0.005407
RMW      -0.002998
CMA      -0.000549
dtype: float64.
GE Model Parameters: const    -0.017100
Mkt-RF    0.011696
SMB      -0.001806
HML       0.001398
RMW      -0.000340
CMA       0.000320
dtype: float64.
IVV Model Parameters: const    -0.018148
Mkt-RF    0.009632
SMB      -0.000578
HML   